In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Shadipur_Delhi_CPCB_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,343.0,177.0,226.0,199.0,319.0,411.0,84.0,65.0,116.0,149.0,302.0,327.0
1,2,355.0,129.0,137.0,193.0,247.0,NaN,94.0,91.0,104.0,177.0,312.0,342.0
2,3,377.0,214.0,136.0,249.0,317.0,NaN,96.0,97.0,70.0,182.0,398.0,324.0
3,4,382.0,354.0,169.0,246.0,346.0,356.0,56.0,69.0,61.0,217.0,395.0,267.0
4,5,343.0,128.0,132.0,248.0,364.0,387.0,76.0,77.0,59.0,182.0,372.0,264.0
5,6,306.0,130.0,141.0,191.0,364.0,247.0,56.0,76.0,88.0,183.0,371.0,281.0
6,7,335.0,177.0,236.0,214.0,403.0,344.0,59.0,64.0,57.0,133.0,391.0,317.0
7,8,380.0,163.0,214.0,272.0,286.0,340.0,59.0,62.0,77.0,195.0,382.0,377.0
8,9,370.0,151.0,147.0,253.0,220.0,273.0,74.0,88.0,112.0,176.0,376.0,320.0
9,10,304.0,342.0,214.0,275.0,229.0,297.0,119.0,88.0,160.0,134.0,331.0,NaN


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,343.000000,177.000000,226.0,199.000000,319.000000,411.000000,84.000000,65.000000,116.000000,149.0,302.000000,327.000000
1,2,355.000000,129.000000,137.0,193.000000,247.000000,209.264706,94.000000,91.000000,104.000000,177.0,312.000000,342.000000
2,3,377.000000,214.000000,136.0,249.000000,317.000000,209.264706,96.000000,97.000000,70.000000,182.0,398.000000,324.000000
3,4,382.000000,354.000000,169.0,246.000000,346.000000,356.000000,56.000000,69.000000,61.000000,217.0,395.000000,267.000000
4,5,343.000000,128.000000,132.0,248.000000,364.000000,387.000000,76.000000,77.000000,59.000000,182.0,372.000000,264.000000
5,6,306.000000,130.000000,141.0,191.000000,364.000000,247.000000,56.000000,76.000000,88.000000,183.0,371.000000,281.000000
6,7,335.000000,177.000000,236.0,214.000000,403.000000,344.000000,59.000000,64.000000,57.000000,133.0,391.000000,317.000000
7,8,380.000000,163.000000,214.0,272.000000,286.000000,340.000000,59.000000,62.000000,77.000000,195.0,382.000000,377.000000
8,9,370.000000,151.000000,147.0,253.000000,220.000000,273.000000,74.000000,88.000000,112.000000,176.0,376.000000,320.000000
9,10,304.000000,342.000000,214.0,275.000000,229.000000,297.000000,119.000000,88.000000,160.000000,134.0,331.000000,260.588235
